In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

In [ ]:
n = 100000

df = pd.DataFrame({
    "transaction_id": range(1, n + 1),

    "transaction_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2026-01-01", "2026-08-31"),
            n
        )
    ),

    "country": np.random.choice(
        ["USA", "Mexico", "India", "UAE", "Poland"],
        n
    ),

    "currency": np.random.choice(
        ["USD", "MXN", "INR", "AED", "PLN"],
        n
    ),

    "transaction_amount": np.round(
        np.random.lognormal(
            mean=5.5,
            sigma=0.8,
            size=n
        ),
        2
    ),

    "transaction_status": np.random.choice(
        ["successful", "failed"],
        n,
        p=[0.965, 0.035]
    ),

    "payment_method": np.random.choice(
        ["bank_transfer", "debit_card", "credit_card", "wallet"],
        n
    ),

    "merchant_category": np.random.choice(
        ["Retail", "Travel", "Utilities", "Food", "Services"],
        n
    )
})

df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

In [ ]:
import sqlite3

conn = sqlite3.connect("analytics.db")

df.to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

In [ ]:
query = """
SELECT
    COUNT(*) AS transaction_count,
    ROUND(SUM(transaction_amount), 2) AS transaction_volume,
    ROUND(AVG(transaction_amount), 2) AS avg_transaction_value,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN transaction_status = 'failed'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS failed_transaction_rate

FROM transactions;
"""

kpis = pd.read_sql_query(query, conn)

kpis

In [ ]:
knowledge = """
FINANCIAL ANALYTICS KNOWLEDGE BASE

Transaction Volume:
The total monetary value of transactions processed during a specified period.

Transaction Count:
The total number of attempted transactions during a specified period.

Average Transaction Value:
Transaction volume divided by transaction count.

Failed Transaction Rate:
The percentage of attempted transactions that did not complete successfully.

A rising failed transaction rate may require investigation into payment channels,
countries, payment methods, technical failures, or operational issues.

Liquidity Risk:
The possibility that an organization may not have enough cash or liquid assets
to meet short-term financial obligations.

Foreign Exchange Exposure:
Financial risk created when changes in currency exchange rates affect the value
of transactions, balances, assets, or liabilities.

ANALYTICAL GUIDELINES

An increase in transaction volume should not automatically be interpreted as
improved performance if transaction failures or operational risk also increase.

Analysts should distinguish observed facts from possible explanations.

Possible drivers should not be presented as confirmed causes without supporting data.
"""

with open("analytics_business_definitions.txt", "w", encoding="utf-8") as f:
    f.write(knowledge)

print("File created successfully.")

In [ ]:
monthly_query = """
SELECT
    SUBSTR(transaction_date, 1, 7) AS month,

    COUNT(*) AS transaction_count,

    ROUND(
        SUM(transaction_amount),
        2
    ) AS transaction_volume,

    ROUND(
        AVG(transaction_amount),
        2
    ) AS avg_transaction_value,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN transaction_status = 'failed'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS failed_transaction_rate

FROM transactions

GROUP BY
    SUBSTR(transaction_date, 1, 7)

ORDER BY month;
"""

monthly_metrics = pd.read_sql_query(
    monthly_query,
    conn
)

monthly_metrics

In [ ]:
current_month = monthly_metrics.iloc[-1]
previous_month = monthly_metrics.iloc[-2]

In [ ]:
metric_context = f"""
CURRENT MONTH

Month: {current_month['month']}
Transaction Count: {current_month['transaction_count']}
Transaction Volume: {current_month['transaction_volume']}
Average Transaction Value: {current_month['avg_transaction_value']}
Failed Transaction Rate: {current_month['failed_transaction_rate']}%

PREVIOUS MONTH

Month: {previous_month['month']}
Transaction Count: {previous_month['transaction_count']}
Transaction Volume: {previous_month['transaction_volume']}
Average Transaction Value: {previous_month['avg_transaction_value']}
Failed Transaction Rate: {previous_month['failed_transaction_rate']}%
"""

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

VECTOR_STORE_ID = os.getenv("OPENAI_VECTOR_STORE_ID")

if not VECTOR_STORE_ID:
    raise RunTimeError("OPENAI_VECTOR_STORE_ID is not configured in .env")

client = OpenAI()

response = client.responses.create(

    model="gpt-5.6-luna",

    instructions="""
    You are a financial analytics assistant.

    Analyze the quantitative metrics supplied by the user.

    Use the analytics knowledge base for metric definitions
    and analytical guidance.

    Clearly distinguish:
    - observed facts
    - potential explanations

    Never invent causes unsupported by the available data.
    """,

    input=f"""
    Analyze the following financial performance metrics.

    {metric_context}

    Provide an executive-level assessment.
    """,

    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [
                VECTOR_STORE_ID
            ],
            "max_num_results": 3
        }
    ]
)

print(response.output_text)

In [ ]:
import json

vector_store = client.vector_stores.retrieve(VECTOR_STORE_ID)

response = client.responses.create(
    model="gpt-5.6-luna",

    instructions="""
    You are a financial analytics assistant.

    Analyze the quantitative metrics supplied by the user.

    Use the analytics knowledge base for metric definitions
    and analytical guidance.

    Clearly distinguish:
    - observed facts
    - potential explanations

    Never invent causes unsupported by the available data.
    """,

    input=f"""
    Analyze the following financial performance metrics.

    {metric_context}

    Provide an executive-level assessment.
    """,

    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [VECTOR_STORE_ID],
            "max_num_results": 3
        }
    ],

    text={
        "format": {
            "type": "json_schema",
            "name": "analytics_report",

            "schema": {
                "type": "object",

                "properties": {
                    "executive_summary": {
                        "type": "string"
                    },

                    "key_findings": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },

                    "risk_flags": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },

                    "possible_drivers": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },

                    "recommended_analysis": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },

                "required": [
                    "executive_summary",
                    "key_findings",
                    "risk_flags",
                    "possible_drivers",
                    "recommended_analysis"
                ],

                "additionalProperties": False
            },

            "strict": True
        }
    }
)

print(response.output_text)

In [ ]:
tools = [

    # RAG / documentation
    {
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
        "max_num_results": 3
    },

    # Monthly KPI data
    {
        "type": "function",
        "name": "get_monthly_kpis",

        "description": """
        Retrieve transaction KPIs for one specified month.
        Use when monthly transaction count, transaction volume,
        average transaction value, or failed transaction rate is needed.
        """,

        "parameters": {
            "type": "object",

            "properties": {
                "month": {
                    "type": "string",
                    "description": "Month in YYYY-MM format."
                }
            },

            "required": ["month"],
            "additionalProperties": False
        },

        "strict": True
    },

    # Country analysis
    {
        "type": "function",
        "name": "get_country_breakdown",

        "description": """
        Retrieve transaction KPIs by country for a specified month.
        Use when the user asks which country performed best or worst,
        geographic risk, country comparison, or country-level
        transaction performance.
        """,

        "parameters": {
            "type": "object",

            "properties": {
                "month": {
                    "type": "string",
                    "description": "Month in YYYY-MM format."
                }
            },

            "required": ["month"],
            "additionalProperties": False
        },

        "strict": True
    }
]

In [ ]:
input_list = [
    {
        "role": "user",
        "content": "What was the failed transaction rate in August 2026?"
    }
]

response = client.responses.create(
    model="gpt-5.6-luna",

    instructions="""
    You are a financial analytics assistant.

    Use available analytical tools whenever data is required.
    Do not invent financial metrics.
    """,

    tools=tools,
    input=input_list
)

In [ ]:
for item in response.output:

    if item.type == "function_call":

        print("Function:", item.name)
        print("Arguments:", item.arguments)
        print("Call ID:", item.call_id)

In [ ]:
DB_PATH = "analytics.db"


def get_monthly_kpis(month):

    query = """
    SELECT
        SUBSTR(transaction_date, 1, 7) AS month,
        COUNT(*) AS transaction_count,
        ROUND(SUM(transaction_amount), 2) AS transaction_volume,
        ROUND(AVG(transaction_amount), 2) AS avg_transaction_value,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN transaction_status = 'failed'
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS failed_transaction_rate
    FROM transactions
    WHERE SUBSTR(transaction_date, 1, 7) = ?
    GROUP BY SUBSTR(transaction_date, 1, 7);
    """

    # Create a NEW connection for this call
    with sqlite3.connect(DB_PATH) as local_conn:

        result = pd.read_sql_query(
            query,
            local_conn,
            params=[month]
        )

    if result.empty:
        return {
            "error": f"No data available for {month}"
        }

    # JSON round-trip converts numpy/pandas values
    # into normal Python values.
    return json.loads(
        result.iloc[0].to_json()
    )

In [ ]:
input_list += response.output

for item in response.output:

    if item.type == "function_call":

        if item.name == "get_monthly_kpis":

            arguments = json.loads(item.arguments)

            result = get_monthly_kpis(
                arguments["month"]
            )

            print("TOOL RESULT:")
            print(result)

            input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(result)
                }
            )

In [ ]:
response = client.responses.create(
    model="gpt-5.6-luna",

    instructions="""
    You are a financial analytics assistant.

    Use available analytical tools whenever data is required.
    Do not invent financial metrics.
    """,

    tools=tools,
    input=input_list
)
print(response.output_text)

In [ ]:
def ask_analytics_assistant(question):

    input_list = [
        {
            "role": "user",
            "content": question
        }
    ]

    response = client.responses.create(
        model="gpt-5.6-luna",

        instructions="""
        You are a financial analytics assistant.

        Use file search for definitions, business rules,
        and analytical guidance.

        Use get_monthly_kpis when actual monthly transaction
        metrics are required.

        Never invent financial metrics.
        If data is unavailable, clearly say so.
        """,

        tools=tools,
        input=input_list
    )

    # Keep going if Astra asks for a custom function
    while True:

        function_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        # No custom function calls remaining:
        # Astra has produced the final response.
        if not function_calls:
            return response

        # Preserve Astra's previous output
        input_list += response.output

        # Execute requested functions
        for item in function_calls:

            print("Tool selected:", item.name)
            print("Arguments:", item.arguments)

            arguments = json.loads(item.arguments)

            if item.name == "get_monthly_kpis":

                result = get_monthly_kpis(
                    arguments["month"]
                )

            else:
                result = {
                    "error": "Unknown function"
                }

            print("Tool result:", result)

            input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(result)
                }
            )

        # Give function result back to Astra
        response = client.responses.create(
            model="gpt-6-astra",

            instructions="""
            You are a financial analytics assistant.

            Use file search for definitions, business rules,
            and analytical guidance.

            Use get_monthly_kpis when actual monthly transaction
            metrics are required.

            Never invent financial metrics.
            """,

            tools=tools,
            input=input_list
        )

In [ ]:
response = ask_analytics_assistant(
    """
    What was the failed transaction rate in August 2026,
    what does that metric mean, and what should an analyst
    investigate when it increases?
    """
)

print(response.output_text)

In [ ]:
AGENT_INSTRUCTIONS = """
You are an AI financial analytics assistant.

Your job is to answer analytical questions using available tools.

Use:
- file_search for definitions, business rules, and analytical guidance.
- get_monthly_kpis for actual month-level transaction metrics.
- get_country_breakdown for country-level financial analysis.

Rules:
1. Never invent numerical results.
2. Use tools whenever external data is needed.
3. You may call multiple tools if required.
4. Compare retrieved results before drawing conclusions.
5. Clearly distinguish observed facts from possible explanations.
6. If required information is unavailable, say so.
"""

In [ ]:
def run_analytics_agent(question, max_steps=6):

    response = client.responses.create(
        model="gpt-6-astra",
        instructions=AGENT_INSTRUCTIONS,
        input=question,
        tools=tools,
        tool_choice="auto",
        parallel_tool_calls=True
    )

    for step in range(max_steps):

        print(f"\n--- Agent Step {step + 1} ---")

        function_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        # If there are no custom function calls left,
        # the agent has finished its work.
        if not function_calls:
            print("No more custom tools required.")
            return response

        tool_outputs = []

        for call in function_calls:

            print("Tool selected:", call.name)
            print("Arguments:", call.arguments)

            arguments = json.loads(call.arguments)

            # ----------------------
            # Execute selected tool
            # ----------------------

            if call.name == "get_monthly_kpis":

                result = get_monthly_kpis(
                    arguments["month"]
                )

            elif call.name == "get_country_breakdown":

                result = get_country_breakdown(
                    arguments["month"]
                )

            else:

                result = {
                    "error": f"Unknown tool: {call.name}"
                }

            print("Tool result:")
            print(result)

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(result)
                }
            )

        # ---------------------------------
        # Return observations to the model
        # ---------------------------------

        response = client.responses.create(
            model="gpt-5.6-luna",

            previous_response_id=response.id,

            instructions=AGENT_INSTRUCTIONS,

            tools=tools,

            tool_choice="auto",

            parallel_tool_calls=True,

            input=tool_outputs
        )

    raise RuntimeError(
        "Agent exceeded maximum allowed steps."
    )

In [ ]:
def get_country_breakdown(month):

    query = """
    SELECT
        country,
        COUNT(*) AS transaction_count,
        ROUND(SUM(transaction_amount), 2) AS transaction_volume,
        ROUND(AVG(transaction_amount), 2) AS avg_transaction_value,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN transaction_status = 'failed'
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS failed_transaction_rate
    FROM transactions
    WHERE SUBSTR(transaction_date, 1, 7) = ?
    GROUP BY country
    ORDER BY failed_transaction_rate DESC;
    """

    # Again: separate connection for every invocation
    with sqlite3.connect(DB_PATH) as local_conn:

        result = pd.read_sql_query(
            query,
            local_conn,
            params=[month]
        )

    if result.empty:
        return {
            "error": f"No data available for {month}"
        }

    return json.loads(
        result.to_json(orient="records")
    )

In [ ]:
response = run_analytics_agent(
    """
    Prepare an executive assessment of August 2026 performance.

    Compare August with July,
    identify any country-level transaction risk,
    explain any important risk metric you use,
    and recommend what the analyst should investigate next.

    Base conclusions only on available data and documentation.
    """
)

print("\nFINAL ANSWER:")
print(response.output_text)

In [ ]:
from agents import (
    Agent, 
    Runner, 
    FileSearchTool, 
    SQLiteSession, 
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    TResponseInputItem,
)

from agents.decorators import tool, input_guardrail

print("Agents SDK imported successfully")

In [ ]:
@tool
def monthly_kpis(month: str) -> str:
    """
    Retrieve transaction KPIs for a specified month.

    Args:
        month: Month in YYYY-MM format, for example 2026-08.
    """

    result = get_monthly_kpis(month)

    return json.dumps(result)


@tool
def country_breakdown(month: str) -> str:
    """
    Retrieve transaction KPIs grouped by country for a specified month.

    Args:
        month: Month in YYYY-MM format, for example 2026-08.
    """

    result = get_country_breakdown(month)

    return json.dumps(result)

In [ ]:
file_search = FileSearchTool(
    vector_store_ids=[VECTOR_STORE_ID],
    max_num_results=3
)

In [ ]:
analytics_agent = Agent(
    name="Financial Analytics Agent",

    model="gpt-5.6-luna",

    instructions="""
    You are a financial analytics agent.

    Your job is to answer analytical questions using the
    available data and documentation.

    Use monthly_kpis for month-level transaction metrics.

    Use country_breakdown for country-level transaction analysis.

    Use file search for business definitions, analytical
    guidelines, and financial metric documentation.

    You may use multiple tools when required.

    Never invent numerical results.

    Clearly distinguish observed facts from possible explanations.

    If the available data cannot support a conclusion,
    explicitly say so.
    """,

    tools=[
        monthly_kpis,
        country_breakdown,
        file_search
    ]
)

In [ ]:
result = await Runner.run(
    analytics_agent,
    """
    Compare July 2026 and August 2026 transaction performance.
    Which month had the higher failed transaction rate?
    """
)

print(result.final_output)

In [ ]:
result = await Runner.run(
    analytics_agent,
    """
    For August 2026, identify the country with the highest
    failed transaction rate.

    Explain what failed transaction rate means and what an analyst
    should investigate when this metric rises.
    """
)

print(result.final_output)

In [ ]:
result = await Runner.run(
    analytics_agent,
    """
    Prepare an executive assessment of August 2026.

    Compare August with July,
    identify country-level transaction risk,
    explain relevant risk metrics,
    and recommend additional analysis.

    Base everything only on available data and documentation.
    """
)

print(result.final_output)

In [ ]:
session = SQLiteSession(
    "analytics_user_001",
    "analytics_conversations.db"
)

In [ ]:
result = await Runner.run(
    analytics_agent,
    """
    Analyze August 2026 transaction performance
    and identify the country with the highest failed transaction rate.
    """,
    session=session
)

print(result.final_output)

In [ ]:
result = await Runner.run(
    analytics_agent,
    """
    Compare that month with the previous month.
    Did the failed transaction rate improve or worsen?
    """,
    session=session
)

print(result.final_output)

In [ ]:
result = await Runner.run(
    analytics_agent,
    """
    What should an analyst investigate based on the risks
    we just identified?
    """,
    session=session
)

print(result.final_output)

In [ ]:
data_agent = Agent(
    name="Data Analysis Specialist",
    model="gpt-6-astra",

    instructions="""
    You are a financial data-analysis specialist.

    Use the available SQL analytics tools to retrieve
    quantitative transaction information.

    Base conclusions only on returned data.
    Never invent metrics.

    Return concise analytical findings to the calling agent.
    """,

    tools=[
        monthly_kpis,
        country_breakdown
    ]
)

In [ ]:
knowledge_agent = Agent(
    name="Business Knowledge Specialist",
    model="gpt-6-astra",

    instructions="""
    You specialize in financial metric definitions,
    business rules, and analytical guidance.

    Use file search to retrieve relevant information
    from the analytics knowledge base.

    Do not invent definitions or policies that are
    not supported by the documentation.
    """,

    tools=[
        file_search
    ]
)

In [ ]:
manager_agent = Agent(
    name="Financial Analytics Manager",
    model="gpt-6-astra",

    instructions="""
    You are the manager of a financial analytics system.

    Delegate quantitative analysis to the data-analysis specialist.

    Delegate metric definitions and analytical guidance to
    the business-knowledge specialist.

    You may call one or both specialists.

    Combine their results into one clear executive response.

    Clearly distinguish observed facts from possible explanations.
    Never invent numerical results.
    """,

    tools=[
        data_agent.as_tool(
            tool_name="data_analysis",
            tool_description="""
            Analyze actual transaction data, monthly KPIs,
            comparisons, and country-level performance.
            """
        ),

        knowledge_agent.as_tool(
            tool_name="business_knowledge",
            tool_description="""
            Retrieve financial definitions, business rules,
            and analytical guidance from documentation.
            """
        )
    ]
)

In [ ]:
result = await Runner.run(
    manager_agent,
    """
    Prepare an executive assessment of August 2026.

    Compare performance with July,
    identify the country with the highest failed transaction rate,
    explain what that metric means,
    and recommend what should be investigated next.
    """
)

print(result.final_output)

In [ ]:
data_handoff_agent = Agent(
    name="Data Analysis Agent",

    handoff_description="""
    Handles questions requiring transaction metrics,
    monthly comparisons, or country-level financial analysis.
    """,

    model="gpt-6-astra",

    instructions="""
    Answer quantitative financial analytics questions
    using the available SQL tools.
    """,

    tools=[
        monthly_kpis,
        country_breakdown
    ]
)

In [ ]:
knowledge_handoff_agent = Agent(
    name="Business Knowledge Agent",

    handoff_description="""
    Handles questions about metric definitions,
    analytical guidance, and business documentation.
    """,

    model="gpt-6-astra",

    instructions="""
    Answer business-definition and analytical-guidance
    questions using file search.
    """,

    tools=[
        file_search
    ]
)

In [ ]:
triage_agent = Agent(
    name="Analytics Triage Agent",

    model="gpt-6-astra",

    instructions="""
    Route the user's question to the appropriate specialist.

    Use the Data Analysis Agent for numerical transaction questions.

    Use the Business Knowledge Agent for definitions,
    business rules, and documentation questions.
    """,

    handoffs=[
        data_handoff_agent,
        knowledge_handoff_agent
    ]
)

In [ ]:
result = await Runner.run(
    triage_agent,
    "Which country had the highest failed transaction rate in August 2026?"
)

print(result.final_output)
print("Answered by:", result.last_agent.name)

In [ ]:
result = await Runner.run(
    triage_agent,
    "What does failed transaction rate mean?"
)

print(result.final_output)
print("Answered by:", result.last_agent.name)

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class AnalyticsScopeCheck(BaseModel):
    in_scope: bool
    reason: str

In [ ]:
scope_guardrail_agent = Agent(
    name="Analytics Scope Guardrail",
    model="gpt-6-astra",

    instructions="""
    Determine whether the user's request belongs to the
    financial analytics assistant.

    IN SCOPE:
    - transaction analysis
    - financial KPIs
    - failed transaction rates
    - country performance
    - transaction volume
    - financial metric definitions
    - analytics recommendations

    OUT OF SCOPE:
    - creative writing
    - recipes
    - unrelated general knowledge
    - entertainment questions

    Return whether the request is in scope.
    """,

    output_type=AnalyticsScopeCheck
)

In [ ]:
@input_guardrail
async def analytics_scope_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:

    result = await Runner.run(
        scope_guardrail_agent,
        input,
        context=ctx.context
    )

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.in_scope
    )

In [ ]:
guarded_manager_agent = manager_agent.clone(
    input_guardrails=[analytics_scope_guardrail]
)

In [ ]:
result = await Runner.run(
    guarded_manager_agent,
    "Compare July and August 2026 failed transaction rates."
)

print(result.final_output)

In [ ]:
try:

    result = await Runner.run(
        guarded_manager_agent,
        "Write me a romantic poem about the ocean."
    )

    print(result.final_output)

except InputGuardrailTripwireTriggered:
    print(
        "Blocked: Request is outside the analytics assistant scope."
    )

In [ ]:
failure_tests = [
    {
        "name": "Missing data",
        "question": "What was the failed transaction rate in January 2027?"
    },

    {
        "name": "Unsupported metric",
        "question": "What was operating profit in August 2026?"
    },

    {
        "name": "Unsupported causality",
        "question": """
        Tell me exactly why failed transaction rate increased
        in August 2026.
        """
    },

    {
        "name": "Out of scope",
        "question": "Write me a poem about summer."
    }
]

In [ ]:
for test in failure_tests:

    print("\nTEST:", test["name"])

    try:

        result = await Runner.run(
            guarded_manager_agent,
            test["question"]
        )

        print(result.final_output)

    except InputGuardrailTripwireTriggered:

        print("GUARDRAIL BLOCKED REQUEST")

In [ ]:
august_kpis = get_monthly_kpis("2026-08")
july_kpis = get_monthly_kpis("2026-07")

print(august_kpis)
print(july_kpis)

In [ ]:
eval_cases = [
    {
        "question":
            "What was the failed transaction rate in August 2026?",

        "expected":
            f"""
            The failed transaction rate should be
            {august_kpis['failed_transaction_rate']}%.
            """
    },

    {
        "question":
            "Compare July and August 2026 failed transaction rates.",

        "expected":
            f"""
            July = {july_kpis['failed_transaction_rate']}%.
            August = {august_kpis['failed_transaction_rate']}%.
            The comparison must be based on those values.
            """
    },

    {
        "question":
            "What does failed transaction rate mean?",

        "expected":
            """
            It should explain that failed transaction rate
            represents the percentage of attempted transactions
            that did not complete successfully.
            """
    },

    {
        "question":
            "What was operating profit in August 2026?",

        "expected":
            """
            The assistant should state that operating-profit
            information is unavailable rather than inventing it.
            """
    }
]

In [ ]:
class EvaluationResult(BaseModel):

    correctness: int = Field(ge=1, le=5)

    groundedness: int = Field(ge=1, le=5)

    relevance: int = Field(ge=1, le=5)

    passed: bool

    notes: str

In [ ]:
evaluation_agent = Agent(
    name="Analytics Agent Evaluator",
    model="gpt-6-astra",

    instructions="""
    Evaluate an AI analytics assistant response.

    Score each category from 1 to 5.

    CORRECTNESS:
    Does the response match the expected facts?

    GROUNDEDNESS:
    Does it avoid unsupported claims and invented numbers?

    RELEVANCE:
    Does it directly answer the user's question?

    Mark passed=true only when the response is
    sufficiently correct, grounded, and relevant.

    Be strict.
    """,

    output_type=EvaluationResult
)

In [ ]:
evaluation_results = []

for case in eval_cases:

    result = await Runner.run(
        guarded_manager_agent,
        case["question"]
    )

    answer = result.final_output

    evaluation_prompt = f"""
    QUESTION:
    {case['question']}

    EXPECTED FACTS / BEHAVIOR:
    {case['expected']}

    ACTUAL ANSWER:
    {answer}
    """

    evaluation = await Runner.run(
        evaluation_agent,
        evaluation_prompt
    )

    score = evaluation.final_output

    evaluation_results.append({
        "question": case["question"],
        "answer": answer,
        "correctness": score.correctness,
        "groundedness": score.groundedness,
        "relevance": score.relevance,
        "passed": score.passed,
        "notes": score.notes
    })

In [ ]:
evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df[
    [
        "correctness",
        "groundedness",
        "relevance",
        "passed",
        "notes"
    ]
]